In [1]:
import sys

!{sys.executable} -m pip uninstall -y torchcrf
!{sys.executable} -m pip install git+https://github.com/kmkurn/pytorch-crf.git
!{sys.executable} -m pip install -q seqeval

Found existing installation: TorchCRF 1.1.0
Uninstalling TorchCRF-1.1.0:
  Successfully uninstalled TorchCRF-1.1.0
  Cloning https://github.com/kmkurn/pytorch-crf.git to /tmp/pip-req-build-_tk7soiu
  Running command git clone --filter=blob:none --quiet https://github.com/kmkurn/pytorch-crf.git /tmp/pip-req-build-_tk7soiu
  Resolved https://github.com/kmkurn/pytorch-crf.git to commit 623e3402d00a2728e99d6e8486010d67c754267b
  Preparing metadata (setup.py) ... done


In [2]:
# =========================================
# 🚀 ICLR-STYLE FUSION NER (PUBLICATION READY)
# =========================================

import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import default_collate
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torchcrf import CRF
from sklearn.model_selection import train_test_split
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

SEED = 42
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5

USE_FUSION = True
USE_CRF = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

labels_list = [
    "O",
    "B-TOPIC", "I-TOPIC",
    "B-METHOD", "I-METHOD",
    "B-ALGORITHM", "I-ALGORITHM",
    "B-CONCEPT", "I-CONCEPT"
]

label2id = {l: i for i, l in enumerate(labels_list)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(labels_list)

sci_tok = AutoTokenizer.from_pretrained(
    "allenai/scibert_scivocab_uncased",
    use_fast=True
)
rob_tok = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

def load_data(path):
    data = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)

            ents = item["output"]
            if isinstance(ents, str):
                ents = json.loads(ents)

            data.append({
                "text": item["input"].lower(),
                "entities": ents
            })
    return data


data = load_data("OS-cleaned_file.jsonl")

train_data, temp = train_test_split(data, test_size=0.2, random_state=SEED)
val_data, test_data = train_test_split(temp, test_size=0.5, random_state=SEED)

def align(text, entities):
    words = text.split()
    labels = ["O"] * len(words)

    for ent in entities:
        ent_words = ent["entity"].split()
        ent_label = ent["label"]

        for i in range(len(words) - len(ent_words) + 1):
            if words[i:i+len(ent_words)] == ent_words:
                labels[i] = "B-" + ent_label
                for j in range(1, len(ent_words)):
                    labels[i+j] = "I-" + ent_label

    return words, labels

class NERDataset(Dataset):
    def __init__(self, data):
        self.samples = []

        for d in data:
            tokens, labels = align(d["text"], d["entities"])
            self.samples.append((tokens, labels))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def encode(tokens, labels):

    sci = sci_tok(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors=None
    )

    rob = rob_tok(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors=None
    )

    word_ids = sci.word_ids()

    label_ids = []
    prev_word = None

    for word_id in word_ids:

        if word_id is None:
            label_ids.append(-100)

        elif word_id != prev_word:
            label_ids.append(label2id.get(labels[word_id], 0))

        else:
            lab = labels[word_id]
            if lab.startswith("B-"):
                lab = lab.replace("B-", "I-")
            label_ids.append(label2id.get(lab, 0))

        prev_word = word_id

    # 🔥 FORCE FIX LENGTHS SAFELY
    def pad(x):
        return (x + [0]*MAX_LEN)[:MAX_LEN]

    return {
        "input_ids": pad(sci["input_ids"]),
        "attention_mask": pad(sci["attention_mask"]),
        "input_ids_rob": pad(rob["input_ids"]),
        "attention_mask_rob": pad(rob["attention_mask"]),
        "labels": pad(label_ids),
        "tokens": tokens
    }

def build_dataset(data):
    enc = []

    for item in data:
        tokens, labels = align(item["text"], item["entities"])

        # skip empty / broken samples (important for stability)
        if len(tokens) == 0:
            continue

        enc.append(encode(tokens, labels))

    return enc


train_enc = build_dataset(train_data)
val_enc   = build_dataset(val_data)
test_enc  = build_dataset(test_data)

test_tokens = [x["tokens"] for x in test_enc]

class TorchDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __getitem__(self, i):
        item = self.data[i]

        return {
            "input_ids": torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "input_ids_rob": torch.tensor(item["input_ids_rob"]),
            "attention_mask_rob": torch.tensor(item["attention_mask_rob"]),
            "labels": torch.tensor(item["labels"])
        }

    def __len__(self):
        return len(self.data)


class FusionNER(nn.Module):
    def __init__(self):
        super().__init__()

        self.sci = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")
        self.rob = AutoModel.from_pretrained("roberta-base")

        hidden = self.sci.config.hidden_size

        # ---------------- FUSION MODULE ----------------
        if USE_FUSION:
            self.gate = nn.Linear(hidden * 2, hidden)

            self.transformer = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(
                    d_model=hidden,
                    nhead=8,
                    batch_first=True
                ),
                num_layers=2
            )

        # ---------------- CLASSIFIER HEAD ----------------
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden, NUM_LABELS)

        # ---------------- CRF LAYER ----------------
        if USE_CRF:
            self.crf = CRF(NUM_LABELS, batch_first=True)

    def forward(self,
                input_ids,
                attention_mask,
                input_ids_rob,
                attention_mask_rob,
                labels=None):

        # ---------------- ENCODERS ----------------
        sci_out = self.sci(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        rob_out = self.rob(
            input_ids=input_ids_rob,
            attention_mask=attention_mask_rob
        ).last_hidden_state

        # ---------------- FUSION ----------------
        if USE_FUSION:
            gate = torch.sigmoid(
                self.gate(torch.cat([sci_out, rob_out], dim=-1))
            )
            fused = gate * sci_out + (1 - gate) * rob_out
            fused = fused + self.transformer(fused)
        else:
            fused = sci_out

        # ---------------- EMISSIONS ----------------
        emissions = self.classifier(self.dropout(fused))
        mask = attention_mask.bool()

        # =========================================================
        # 🧠 CRF MODE
        # =========================================================
        if USE_CRF:

            # ---------------- INFERENCE MODE ----------------
            if labels is None:
                return self.crf.decode(emissions, mask=mask)

            # ---------------- TRAINING MODE ----------------
            labels = labels.clone()
            labels[labels == -100] = 0

            loss = -self.crf(
                emissions,
                labels,
                mask=mask,
                reduction='mean'
            )
            return loss

        # =========================================================
        # 🧠 NON-CRF MODE
        # =========================================================
        return emissions.argmax(-1)

model = FusionNER().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

train_loader = DataLoader(
    TorchDataset(train_enc),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=default_collate
)
val_loader = DataLoader(
    TorchDataset(val_enc),
    batch_size=BATCH_SIZE,
    collate_fn=default_collate
)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        loss = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE),
            batch["input_ids_rob"].to(DEVICE),
            batch["attention_mask_rob"].to(DEVICE),
            batch["labels"].to(DEVICE)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")

def evaluate(loader):
    model.eval()

    preds, trues = [], []

    # ✅ For accuracy
    total_tokens = 0
    correct_tokens = 0

    with torch.no_grad():
        for batch in loader:

            outputs = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE),
                batch["input_ids_rob"].to(DEVICE),
                batch["attention_mask_rob"].to(DEVICE),
                labels=None
            )

            labels = batch["labels"].cpu().numpy()

            for i in range(len(labels)):

                true_seq = []
                pred_seq = []

                pred = outputs[i]

                for j in range(len(labels[i])):

                    if labels[i][j] == -100:
                        continue

                    true_label = id2label[labels[i][j]]

                    pred_id = pred[j] if USE_CRF else int(pred[j])
                    pred_label = id2label[pred_id]

                    # ✅ collect for seqeval
                    true_seq.append(true_label)
                    pred_seq.append(pred_label)

                    # ✅ compute accuracy
                    total_tokens += 1
                    if true_label == pred_label:
                        correct_tokens += 1

                preds.append(pred_seq)
                trues.append(true_seq)

    accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0

    return {
        "accuracy": accuracy,  # 🔥 ADDED
        "precision": precision_score(trues, preds),
        "recall": recall_score(trues, preds),
        "f1": f1_score(trues, preds),
        "report": classification_report(trues, preds)
    }

test_loader = DataLoader(TorchDataset(test_enc), batch_size=BATCH_SIZE)

results = evaluate(test_loader)

print("\n🔥 FINAL TEST RESULTS")
print(results)

torch.save(model.state_dict(), "fusion_iclr_model.pt")

with open("iclr_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("✅ ICLR-STYLE MODEL SAVED")
import json

def extract_entities_from_sequence(text_tokens, pred_labels):

    entities = []
    current_entity = []
    current_label = None

    for token, label in zip(text_tokens, pred_labels):

        if label == "O":
            if current_entity:
                entities.append({
                    "entity": " ".join(current_entity),
                    "label": current_label.replace("B-", "").replace("I-", "")
                })
                current_entity = []
                current_label = None
            continue

        if label.startswith("B-"):
            if current_entity:
                entities.append({
                    "entity": " ".join(current_entity),
                    "label": current_label.replace("B-", "").replace("I-", "")
                })

            current_entity = [token]
            current_label = label

        elif label.startswith("I-") and current_entity:
            current_entity.append(token)

    if current_entity:
        entities.append({
            "entity": " ".join(current_entity),
            "label": current_label.replace("B-", "").replace("I-", "")
        })

    return entities

def generate_test_predictions(loader, raw_texts):
    model.eval()
    results = []

    idx = 0

    with torch.no_grad():
        for batch in loader:

            outputs = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE),
                batch["input_ids_rob"].to(DEVICE),
                batch["attention_mask_rob"].to(DEVICE),
                labels=None
            )

            for i in range(len(outputs)):

                pred_ids = outputs[i]

                text = raw_texts[idx]["text"]
                tokens = text.split()

                # 🔥 SAME TOKENIZATION AS TRAINING
                enc = sci_tok(
                    tokens,
                    is_split_into_words=True,
                    truncation=True,
                    padding="max_length",
                    max_length=MAX_LEN,
                    return_tensors=None
                )

                word_ids = enc.word_ids()

                # =========================================
                # ✅ FIX: SUBWORD → WORD LEVEL ALIGNMENT
                # =========================================
                word_level_preds = {}

                for j, word_id in enumerate(word_ids):

                    if word_id is None:
                        continue
                    if j >= len(pred_ids):
                        continue

                    pred_id = pred_ids[j] if USE_CRF else int(pred_ids[j])
                    label = id2label[pred_id]

                    # 🔥 take ONLY first subword prediction
                    if word_id not in word_level_preds:
                        word_level_preds[word_id] = label

                # reconstruct aligned labels
                pred_labels = []
                for w_idx in range(len(tokens)):
                    if w_idx in word_level_preds:
                        pred_labels.append(word_level_preds[w_idx])
                    else:
                        pred_labels.append("O")  # safe fallback

                # =========================================
                # BIO → ENTITY CONVERSION
                # =========================================
                entities = extract_entities_from_sequence(tokens, pred_labels)

                results.append({
                    "text": text,
                    "entities": entities
                })

                idx += 1

    return results

# =========================
# RUN ON TEST SET
# =========================

test_loader = DataLoader(TorchDataset(test_enc), batch_size=BATCH_SIZE)

# raw test texts
raw_test_texts = test_data  # already loaded before encoding

predictions = generate_test_predictions(test_loader, raw_test_texts)

# =========================
# SAVE TO JSON FILE
# =========================

with open("test_predictions.json", "w") as f:
    json.dump(predictions, f, indent=4)

print("✅ Predictions saved to test_predictions.json")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Epoch 4 | Loss: 214.2997


100%|██████████| 157/157 [01:37<00:00,  1.60it/s]


Epoch 5 | Loss: 133.0051


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



🔥 FINAL TEST RESULTS
{'accuracy': 0.9910640066500416, 'precision': np.float64(0.8622754491017964), 'recall': np.float64(0.8372093023255814), 'f1': np.float64(0.8495575221238938), 'report': '              precision    recall  f1-score   support\n\n   ALGORITHM       0.00      0.00      0.00         1\n     CONCEPT       0.87      0.88      0.88       262\n      METHOD       0.80      0.44      0.57        27\n       TOPIC       0.83      0.83      0.83        54\n\n   micro avg       0.86      0.84      0.85       344\n   macro avg       0.63      0.54      0.57       344\nweighted avg       0.86      0.84      0.84       344\n'}
✅ ICLR-STYLE MODEL SAVED
✅ Predictions saved to test_predictions.json


In [ ]:
# =========================================
# 1️⃣ Install Libraries
# =========================================
!pip install -q transformers datasets scikit-learn

# =========================================
# 2️⃣ Imports
# =========================================
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    logging
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

logging.set_verbosity_error()

# =========================================
# 3️⃣ Load RE Dataset
# =========================================
with open("/content/OS_relations_fixed.json") as f:
    data = json.load(f)

texts = [d["input"] for d in data]
labels = [d["output"].strip() for d in data]

label_list = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

labels = [l if l in label2id else "no_relation" for l in labels]

# =========================================
# 4️⃣ Train / Val / Test Split
# =========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

def make_dataset(texts, labels):
    return Dataset.from_dict({
        "text": texts,
        "label": [label2id[l] for l in labels]
    })

train_dataset = make_dataset(train_texts, train_labels)
val_dataset = make_dataset(val_texts, val_labels)
test_dataset = make_dataset(test_texts, test_labels)

# =========================================
# 5️⃣ Tokenization
# =========================================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

for d in [train_dataset, val_dataset, test_dataset]:
    d.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =========================================
# 6️⃣ Model
# =========================================
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# =========================================
# 7️⃣ Metrics
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# =========================================
# 8️⃣ Training Args
# =========================================
training_args = TrainingArguments(
    output_dir="./roberta-relation",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# =========================================
# 9️⃣ Trainer
# =========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =========================================
# 🔟 Train
# =========================================
trainer.train()

# =========================================
# 🧪 TEST SET EVALUATION
# =========================================
test_metrics = trainer.evaluate(eval_dataset=test_dataset)

print("\n📊 TEST SET RESULTS:")
print(f"Accuracy : {test_metrics['eval_accuracy']:.4f}")
print(f"Precision: {test_metrics['eval_precision']:.4f}")
print(f"Recall   : {test_metrics['eval_recall']:.4f}")
print(f"F1 Score : {test_metrics['eval_f1']:.4f}")

# =========================================
# 1️⃣1️⃣ Load NER Output
# =========================================
with open("/content/FINAL_KG_READY.json") as f:
    test_data = json.load(f)

# =========================================
# 1️⃣2️⃣ Clean Entities
# =========================================
def clean_entities(entities):
    cleaned = []
    for e in entities:
        text = e["entity"].strip().lower()

        if len(text) < 4:
            continue
        if text in ["the", "this", "that", "data"]:
            continue
        if e["label"] == "OTHER":
            continue

        cleaned.append(e)

    return cleaned[:5]   # 🔥 LIMIT TO TOP 5

# =========================================
# 1️⃣3️⃣ Direction-Aware Prediction
# =========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def predict_best_relation(e1, t1, e2, t2, text):
    def run(inp):
        inputs = tokenizer(inp, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model(**inputs)
            probs = torch.softmax(out.logits, dim=1)
            conf, pred = torch.max(probs, dim=1)
        return id2label[pred.item()], conf.item()

    input1 = f"Sentence: {text} Entity1: {e1} ({t1}). Entity2: {e2} ({t2})."
    input2 = f"Sentence: {text} Entity1: {e2} ({t2}). Entity2: {e1} ({t1})."

    r1, c1 = run(input1)
    r2, c2 = run(input2)

    if c1 >= c2:
        return e1, t1, r1, e2, t2, c1
    else:
        return e2, t2, r2, e1, t1, c2

# =========================================
# 1️⃣4️⃣ Predict Relations
# =========================================
triplets = []
seen = set()

for item in test_data:
    text = item["text"]
    entities = clean_entities(item.get("entities", []))

    for i in range(len(entities)):
        for j in range(i+1, len(entities)):

            e1 = entities[i]["entity"]
            t1 = entities[i]["label"]
            e2 = entities[j]["entity"]
            t2 = entities[j]["label"]

            if e1.lower() == e2.lower():
                continue

            e1, t1, rel, e2, t2, conf = predict_best_relation(e1, t1, e2, t2, text)

            # 🔥 Strong filtering
            if rel == "no_relation" or conf < 0.8:
                continue

            key = (e1.lower(), rel, e2.lower())
            if key in seen:
                continue
            seen.add(key)

            triplets.append({
                "text": text,
                "entity1": e1,
                "entity1_type": t1,
                "relation": rel,
                "entity2": e2,
                "entity2_type": t2,
                "confidence": round(conf, 3)
            })

# =========================================
# 1️⃣5️⃣ Save Output
# =========================================
with open("final_triplets_level2.json", "w") as f:
    json.dump(triplets, f, indent=4)

print("\n✅ FINAL LEVEL-2 triplets saved to final_triplets_level2.json")

Map:   0%|          | 0/1761 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'eval_loss': '0.9113', 'eval_accuracy': '0.7041', 'eval_precision': '0.643', 'eval_recall': '0.7041', 'eval_f1': '0.6581', 'eval_runtime': '1.369', 'eval_samples_per_second': '71.56', 'eval_steps_per_second': '9.493', 'epoch': '1'}


...

{'eval_loss': '0.5876', 'eval_accuracy': '0.8061', 'eval_precision': '0.7839', 'eval_recall': '0.8061', 'eval_f1': '0.788', 'eval_runtime': '1.338', 'eval_samples_per_second': '73.24', 'eval_steps_per_second': '9.715', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.5921', 'grad_norm': '13.23', 'learning_rate': '1.919e-06', 'epoch': '4.525'}
{'eval_loss': '0.5162', 'eval_accuracy': '0.8469', 'eval_precision': '0.8316', 'eval_recall': '0.8469', 'eval_f1': '0.8376', 'eval_runtime': '1.401', 'eval_samples_per_second': '69.94', 'eval_steps_per_second': '9.278', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '691.9', 'train_samples_per_second': '12.72', 'train_steps_per_second': '1.597', 'train_loss': '0.7418', 'epoch': '5'}
{'eval_loss': '0.7119', 'eval_accuracy': '0.8061', 'eval_precision': '0.8088', 'eval_recall': '0.8061', 'eval_f1': '0.7873', 'eval_runtime': '1.301', 'eval_samples_per_second': '75.31', 'eval_steps_per_second': '9.99', 'epoch': '5'}

📊 TEST SET RESULTS:
Accuracy : 0.8061
Precision: 0.8088
Recall   : 0.8061
F1 Score : 0.7873

✅ FINAL LEVEL-2 triplets saved to final_triplets_level2.json
